# Lab 3 — Rain Over Europe: Analyse the Past

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-eda.ipynb)

Six years of hourly weather for 45 European cities — the 27 capitals
the feed covers and 18 large cities — from 2020-01-01 to 2026-03-03.
Challenge 177, *European Rain Forecast*, asks for the rain
at each of them **6 hours** and **48 hours** after the forecast is
made, as a distribution, scored with the CRPS once the rain has
fallen.

Before any model, read the past. Each section below asks one question
a forecaster asks, answers it with a number, and leaves you one
question to answer in a text cell. The last section scores four
simple forecasts with the challenge's own metric, on a period no
model has seen.

**Time:** 25 minutes, the ten answers included. **Deliverable:**
your copy run end to end, the ten answers, and the table of
section 9.

It needs pandas, numpy and matplotlib, and runs in about two minutes
on Colab.

---

## 0. Setup

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile →
API Keys (it starts with `mlk_user_`). Never paste its value into a
cell: a notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and
allow this notebook to access it. Outside Colab, set it in your
environment before starting Jupyter; the cell raises if it is missing.
Colab already has pandas, numpy and matplotlib; the only install is
the ML-Arena client.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlarena-sdk"], check=True)
    from google.colab import userdata
    os.environ["MLARENA_API_KEY"] = userdata.get("MLARENA_API_KEY")

import matplotlib.pyplot as plt
import mlarena
import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 120)
client = mlarena.connect(api_key=os.environ["MLARENA_API_KEY"])
CHALLENGE_ID = 177

Challenge 177 carries two datasets: the 28 MB European panel this lab
uses, and seven yearly files for ~990 cities worldwide (4.4 GB).
`client.download_dataset(177)` would fetch all of them, so the cell
below asks for the list, picks the one file by its label, and fetches
its signed `download_url` (valid one hour, no key needed).

In [ ]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = client.datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

---

## 1. Load and check

Timestamps are UTC in the file (`2025-01-01 00:00:00+00`): parse them
as such, never as local time, or every hour-of-day below shifts. The
city and country columns repeat 45 values 54,096 times each; as
categories they cost a byte per row instead of a Python string.

In [ ]:
df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")

hours = pd.date_range(df["timestamp"].min(), df["timestamp"].max(),
                      freq="h")
n_cities = df["city_name"].nunique()
print(df.shape, df["timestamp"].min(), "->", df["timestamp"].max())
print(f"{len(hours)} hours x {n_cities} cities = "
      f"{len(hours) * n_cities} rows expected")
assert not df.duplicated(["timestamp", "city_name"]).any()
assert len(df) == len(hours) * n_cities
print(df.isna().sum()[lambda s: s > 0])
df.head(3)

Wide tables make everything below one line of numpy: one row per hour,
one column per city. `W` is the wet/dry indicator the whole notebook
is about: a city-hour is **wet** at 0.1 mm or more.

In [ ]:
def wide(col):
    out = df.pivot(index="timestamp", columns="city_name", values=col)
    out.columns = out.columns.astype(str)
    return out

rain, hum, clouds = wide("rain"), wide("humidity"), wide("clouds")
cities = (df.groupby("city_name", observed=True)
          [["country_code", "latitude", "longitude"]].first())
cities.index = cities.index.astype(str)
cities = cities.loc[rain.columns]

W = (rain >= 0.1).to_numpy()          # (hours, cities) bool
month = rain.index.month.to_numpy()
utc_hour = rain.index.hour.to_numpy()
print(rain.shape, W.dtype)

**Finding.** 2,434,320 rows = 54,096 hours × 45 cities: no hour and
no city is missing, and no row is duplicated. The only empty column is
`visibility`, empty for every row — the feed does not report it. The
challenge still sends it in the agent's input, as 0.

**Question 1.** The live feed *does* drop whole hours now and then; the
challenge carries the previous hour forward and flags it in
`observed`. Why does it index hours by timestamp and not by position
when it computes the hour `issue_time + 6`?

*Your answer:*

---

## 2. The panel

A scatter of longitude and latitude is a map, once the x axis is
shrunk by cos(latitude): a degree of longitude at 50°N is 0.64 of a
degree of latitude.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.scatter(cities["longitude"], cities["latitude"], s=18,
           color="#2a78d6")
for name, row in cities.iterrows():
    ax.annotate(name, (row["longitude"], row["latitude"]),
                xytext=(3, 3), textcoords="offset points", fontsize=7)
ax.set_aspect(1 / np.cos(np.radians(50)))
ax.set_xlabel("longitude (°E)")
ax.set_ylabel("latitude (°N)")
ax.set_title("The 45 cities of the panel")
plt.show()

print(cities.sort_values("longitude").round(2).to_string())

**Finding.** From Dublin (−6.3°E) to Moscow (37.6°E) and from Sevilla
(37.4°N) to Helsinki and Saint Petersburg (60°N): Atlantic, North Sea,
Baltic, Mediterranean and continental climates. The panel is dense in
the centre (Germany, the Low Countries, northern Italy) and thin at
the edges: nothing west of Dublin, nothing in the Atlantic.

**Question 2.** Weather over Europe mostly arrives from the west
(section 7). Which cities have no neighbour upwind of them in the
panel, and what does that mean for a model that uses the rain in
other cities as a feature?

*Your answer:*

---

## 3. Zero inflation

Most hours are dry. How many, where, and how much falls when it does?

In [ ]:
wet_frac = pd.Series(W.mean(axis=0), index=rain.columns)
print(f"wet city-hours overall: {W.mean():.1%}")
print("driest:", wet_frac.nsmallest(5).round(3).to_dict())
print("wettest:", wet_frac.nlargest(5).round(3).to_dict())

amounts = rain.to_numpy()[W]              # wet hours only
q = np.quantile(amounts, [0.5, 0.9, 0.99])
print(f"wet-hour amount: median {q[0]:.1f} mm, p90 {q[1]:.1f}, "
      f"p99 {q[2]:.1f}, max {amounts.max():.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sc = ax1.scatter(cities["longitude"], cities["latitude"],
                 c=wet_frac, cmap="Blues", s=40, vmin=0,
                 edgecolors="#52514e", linewidths=0.4)
ax1.set_aspect(1 / np.cos(np.radians(50)))
fig.colorbar(sc, ax=ax1, label="share of wet hours")
ax1.set_title("How often it rains")
ax2.hist(amounts, bins=np.arange(0.05, 25.1, 0.1),
         color="#2a78d6")
ax2.set_yscale("log")
ax2.set_xlabel("rain in a wet hour (mm)")
ax2.set_ylabel("city-hours (log scale)")
ax2.set_title("How much, when it does")
plt.show()

The file stores rain to one decimal. Look at the values themselves:

In [ ]:
values, counts = np.unique(np.round(amounts, 1), return_counts=True)
share = pd.Series(counts / counts.sum(), index=values)
print(f"{len(values)} distinct wet values, the first ones:")
print(share.head(6).round(3).to_string())
r = rain.to_numpy()
print("hours with 0 < rain < 0.1 mm:", int(((r > 0) & (r < 0.1)).sum()))

**Finding.** 14.5% of city-hours are wet — from 7.9% in Athens to
28.3% in Glasgow, a gradient from the dry south-east to the wet
north-west. When it rains, it mostly drizzles: the median wet hour is
0.2 mm, the 90th percentile 1.4 mm, the 99th 4.5 mm, the maximum
24.9 mm — a hundred times fewer hours at 6 mm than at 1 mm, and
still a few at 20.
Amounts are multiples of 0.1 mm, with nothing between 0 and 0.1:
37% of wet hours are exactly 0.1 mm and 15% exactly 0.2 mm.

So the distribution of the target is a spike at 0 holding ~85% of the
mass, a second pile at 0.1–0.2 mm, and a long right tail. Its median
is 0 almost everywhere, almost always.

**Question 3.** A deterministic forecast scored with absolute error is
best at the median. What does that make the best single number to
forecast here, and why must a forecast *distribution* still put some
members above zero?

*Your answer:*

---

## 4. Seasonality

Wet frequency per city and calendar month. The rows are sorted by the
ratio of the summer (June–August) to the winter (December–February)
frequency, so the cities with dry summers sit at the top. The left
panel is the frequency itself; the right one divides each row by the
city's own yearly mean, so the *shape* of the year shows even where
the level is low (brown: drier than usual, green: wetter).

In [ ]:
from matplotlib.colors import LogNorm

by_month = pd.DataFrame(W, index=rain.index,
                        columns=rain.columns).groupby(month).mean()
summer_winter = (by_month.loc[[6, 7, 8]].mean()
                 / by_month.loc[[12, 1, 2]].mean()).sort_values()
heat = by_month[summer_winter.index].T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 10), sharey=True)
im1 = ax1.imshow(heat, cmap="Blues", aspect="auto", vmin=0)
im2 = ax2.imshow(heat.div(heat.mean(axis=1), axis=0), cmap="BrBG",
                 aspect="auto", norm=LogNorm(vmin=1 / 3, vmax=3))
ax1.set_yticks(range(len(heat)), heat.index, fontsize=7)
for ax in (ax1, ax2):
    ax.set_xticks(range(12), list("JFMAMJJASOND"))
fig.colorbar(im1, ax=ax1, label="share of wet hours")
cb = fig.colorbar(im2, ax=ax2, label="relative to the city's mean")
cb.set_ticks([1 / 3, 1 / 2, 1, 2, 3], labels=["1/3", "1/2", "1", "2",
                                          "3"])
cb.minorticks_off()
ax1.set_title("Wet frequency")
ax2.set_title("Relative to the city's own mean")
plt.show()

extremes = pd.concat([summer_winter.head(5), summer_winter.tail(3)])
print("summer / winter:", extremes.round(2).to_dict())
print("panel by month:", by_month.mean(axis=1).round(3).to_dict())

Frequency is one half of the story; the amount that falls in a wet
hour is the other.

In [ ]:
SEASONS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8],
           "SON": [9, 10, 11]}
rows = {}
for name, months in SEASONS.items():
    sel = np.isin(month, months)
    wet_amounts = r[sel][W[sel]]
    rows[name] = {
        "wet share": W[sel].mean(),
        "median mm": np.median(wet_amounts),
        "p90 mm": np.quantile(wet_amounts, 0.9),
        "p99 mm": np.quantile(wet_amounts, 0.99),
        "mean mm/h": r[sel].mean(),
    }
print(pd.DataFrame(rows).T.round(3))

**Finding.** Averaged over the panel, the season barely moves how
*often* it rains (13–17% in every month) — because the panel holds two
opposite regimes that cancel. Sevilla's summer is 0.12 times as wet as
its winter, Palermo's 0.24, Athens' and Madrid's 0.27: the Mediterranean
dries out. Moscow's summer is 4.0 times as wet as its winter, Saint
Petersburg's 2.3: the continental interior gets its rain in summer.
The season moves *how much* falls: a wet summer hour has a 99th
percentile of 5.7 mm against 3.4 mm in winter — convective showers
are short and heavy, winter fronts long and light.

**Question 4.** A model that knows the month but not the city would
learn almost nothing from it here. Which two columns does the month
need beside it to be useful?

*Your answer:*

---

## 5. The diurnal cycle

Convection follows the sun, not the clock in London. The local *solar*
hour is the UTC hour plus longitude / 15 (the sun moves 15° an hour):
noon in Moscow is 2.5 hours before noon in Dublin, whatever the time
zones say. Summer wet frequency by local solar hour, each city divided
by its own daily mean so the shapes compare:

In [ ]:
lon = cities["longitude"].to_numpy()
solar_hour = np.floor(utc_hour[:, None] + lon[None, :] / 15 + 0.5) % 24

def diurnal(months):
    """(24, cities) wet frequency by local solar hour."""
    sel = np.isin(month, months)
    out = np.zeros((24, W.shape[1]))
    for j in range(W.shape[1]):
        s = pd.Series(W[sel, j]).groupby(solar_hour[sel, j]).mean()
        out[:, j] = s.reindex(range(24)).to_numpy()
    return pd.DataFrame(out, columns=rain.columns)

summer = diurnal([6, 7, 8])
winter = diurnal([12, 1, 2])

def afternoon_over_night(d):
    return d.loc[12:17].mean() / d.loc[0:5].mean()

ratio = afternoon_over_night(summer).sort_values()
print("summer, afternoon (12-17 h) / night (0-5 h), solar time")
print(" flattest:", ratio.head(4).round(2).to_dict())
print(" steepest:", ratio.tail(4).round(2).to_dict())
print(" maritime:", ratio[["Dublin", "Glasgow", "London"]]
      .round(2).to_dict())
print("winter, same ratio: "
      f"{afternoon_over_night(winter).min():.2f} to "
      f"{afternoon_over_night(winter).max():.2f}")

SHOW = {"Athens": "#eb6834", "Moscow": "#2a78d6",
        "Dublin": "#1baf7a"}
fig, ax = plt.subplots(figsize=(8, 4))
for name, colour in SHOW.items():
    y = summer[name] / summer[name].mean()
    ax.plot(range(24), y, color=colour, lw=2, label=name)
ax.axhline(1, color="#8c8b86", lw=1, ls=":")
ax.set_xlabel("local solar hour")
ax.set_ylabel("wet frequency / daily mean")
ax.set_title("Summer (June-August)")
ax.set_xticks(range(0, 24, 3))
ax.legend()
plt.show()

**Finding.** In summer the rain comes in the afternoon almost
everywhere: between 12 and 17 h solar time a city is wet 1.05
(Barcelona) to 10.5 (Athens) times as often as between 0 and 5 h.
The continental east and Italy are steep (Kharkiv 2.9, Moscow 3.0,
Rome 3.1) — convection that builds as the ground heats. The British
cities have a milder peak (Glasgow 1.7, Dublin 2.2, London 2.6), and
the western Mediterranean coast almost none (Barcelona 1.05,
Marseille 1.14, Valencia 1.22). In winter the cycle is gone: the
same ratio runs from 0.78 to 1.39.

**Question 5.** The challenge issues forecasts at fixed UTC hours.
Is `issue_time.hour` the right feature for the diurnal cycle of all
45 cities? What would you compute instead?

*Your answer:*

---

## 6. Persistence and memory

If it is raining now, how likely is rain *h* hours later? Pooled over
the 45 cities, for *h* from 1 to 72, against the chance of rain at a
random hour, and against the chance after a dry hour:

In [ ]:
p_wet = W.mean()
lags = np.arange(1, 73)
after_wet, after_dry = [], []
for h in lags:
    now, later = W[:-h], W[h:]
    after_wet.append((now & later).sum() / now.sum())
    after_dry.append((~now & later).sum() / (~now).sum())
memory = pd.DataFrame({"P(wet | wet now)": after_wet,
                       "P(wet | dry now)": after_dry}, index=lags)
print(memory.loc[[1, 3, 6, 12, 24, 36, 48, 72]].round(3))
print(f"P(wet) at a random hour: {p_wet:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lags, after_wet, color="#2a78d6", lw=2,
        label="after a wet hour")
ax.plot(lags, after_dry, color="#eb6834", lw=2,
        label="after a dry hour")
ax.axhline(p_wet, color="#8c8b86", lw=1, ls=":",
           label="climatology")
for h in (6, 48):
    ax.axvline(h, color="#52514e", lw=1, ls="--")
    ax.annotate(f"+{h} h", (h, 0.6), xytext=(4, 0),
                textcoords="offset points")
ax.set_xlabel("hours later (h)")
ax.set_ylabel("P(wet at t + h)")
ax.set_xticks([1, 6, 12, 24, 36, 48, 60, 72])
ax.legend()
plt.show()

Part of what survives at +48 h is not memory at all: a wet hour is
more likely to come from a wet city in a wet month, which stays wet
anyway. Compare instead with the climatology of the *same city and
month* as the hour being forecast:

In [ ]:
clim = by_month.to_numpy()                 # (12, cities)
rows = {}
for h in (1, 6, 12, 24, 48, 72):
    now, later = W[:-h], W[h:]
    expected = clim[month[h:] - 1]         # same city, same month
    rows[h] = {
        "lift | wet now": (now & later).sum()
        / (expected * now).sum(),
        "lift | dry now": (~now & later).sum()
        / (expected * ~now).sum(),
    }
print(pd.DataFrame(rows).T.rename_axis("h").round(2))

**Finding.** An hour after a wet hour, the next is wet 73% of the
time; 6 hours after, still 43% — three times the 14.5% of a random
hour; 48 hours after, 23%. After a dry hour the curve climbs back
from below: 10% at +6 h, 13% at +48 h. Against the climatology of
the same city and month, a wet hour now multiplies the chance of rain
at +6 h by 2.6 and a dry one by 0.68; at +48 h, by 1.41 and 0.92.

Since 85% of issue hours are dry, the best +48 h forecast is most of
the time the city's climatology for that month, times 0.92. **That is
the point of +48 h:** the skill there comes from getting the
climatology right — per city, per month, with the right shape — not
from the last hours. At +6 h the state at the issue time matters.

The curve is not monotone: it flattens at 24 h and rises again
towards 48 h, the same time of day — the diurnal cycle of section 5.

**Question 6.** At +48 h, which input does a good forecast mostly
rest on, and which one only nudges it? At +6 h?

*Your answer:*

---

## 7. Spatial structure

How much does rain in one city say about rain in another at the same
hour? For every pair, the lift P(wet in B | wet in A) / P(wet in B),
against the distance between them (great circle):

In [ ]:
lat_r = np.radians(cities["latitude"].to_numpy())
lon_r = np.radians(lon)
a = (np.sin((lat_r[:, None] - lat_r[None, :]) / 2) ** 2
     + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :])
     * np.sin((lon_r[:, None] - lon_r[None, :]) / 2) ** 2)
dist_km = 2 * 6371 * np.arcsin(np.sqrt(a))

Wf = W.astype(np.float32)
both = Wf.T @ Wf                             # wet together, hours
lift = both / W.sum(axis=0)[:, None] / W.mean(axis=0)[None, :]
iu = np.triu_indices(len(cities), k=1)
pairs = pd.DataFrame({"km": dist_km[iu], "lift": lift[iu]})
bins = [0, 300, 600, 1000, 1500, 2000, 4000]
binned = pairs.groupby(pd.cut(pairs["km"], bins),
                       observed=True)["lift"].agg(["mean", "count"])
print(binned.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(pairs["km"], pairs["lift"], s=8, alpha=0.4,
           color="#2a78d6")
ax.axhline(1, color="#8c8b86", lw=1, ls=":")
ax.set_xlabel("distance between the two cities (km)")
ax.set_ylabel("lift of wet together")
ax.set_title("All 990 pairs, same hour")
plt.show()

Same hour is not the whole story. Weather systems move, mostly
eastward, so rain in London *now* should say something about rain
further east *later*. A plain correlation over lags would be
dominated by the daily and yearly cycles every city shares, so
correlate the **anomalies**: each city's wet indicator minus its own
mean for that month and UTC hour.

In [ ]:
wet_df = pd.DataFrame(Wf, index=rain.index, columns=rain.columns)
anom = wet_df - wet_df.groupby([month, utc_hour]).transform("mean")
A = anom.to_numpy()

def lagged_corr(src, dst, lags=range(-24, 49)):
    """corr(src at t, dst at t + lag), for each lag in hours."""
    x = (A[:, src] - A[:, src].mean()) / A[:, src].std()
    y = (A[:, dst] - A[:, dst].mean()) / A[:, dst].std()
    n = len(x)
    return pd.Series({L: np.mean(x[max(0, -L):n - max(0, L)]
                                 * y[max(0, L):n + min(0, L)])
                      for L in lags})

col = {name: j for j, name in enumerate(rain.columns)}
CHAIN = [("Dublin", "London"), ("London", "Amsterdam"),
         ("Amsterdam", "Berlin"), ("Berlin", "Warsaw"),
         ("Warsaw", "Minsk"), ("Minsk", "Moscow"),
         ("London", "Berlin"), ("London", "Warsaw")]
curves = {}
for src, dst in CHAIN:
    c = lagged_corr(col[src], col[dst])
    curves[(src, dst)] = c
    print(f"{src:>9} -> {dst:<9} {dist_km[col[src], col[dst]]:5.0f} km"
          f"  peak at {c.idxmax():+3d} h  r = {c.max():.3f}"
          f"  (r at 0 h: {c[0]:.3f})")

fig, ax = plt.subplots(figsize=(8, 4))
for dst, colour in (("Amsterdam", "#1baf7a"), ("Berlin", "#2a78d6"),
                    ("Warsaw", "#eb6834")):
    c = curves[("London", dst)]
    ax.plot(c.index, c.values, color=colour, lw=2, label=dst)
    ax.axvline(c.idxmax(), color=colour, lw=1, ls="--")
ax.axhline(0, color="#8c8b86", lw=1)
ax.set_xlabel("lag (h): London at t, the other city at t + lag")
ax.set_ylabel("correlation of anomalies")
ax.set_title("Rain in London, then further east")
ax.legend()
plt.show()

**Finding.** At the same hour, two cities under 300 km apart are wet
together 2.7 times as often as chance would have it; 1.9 times at
300–600 km, 1.3 at 600–1,000 km, and no more than chance beyond about
1,500 km. Over time, the rain moves east. London's anomaly correlates
best with Amsterdam's 6 hours later (357 km), Berlin's 22 hours later
(932 km) and Warsaw's 38 hours later (1,448 km), and the peak flattens
as it goes. Step by step, Dublin → London → Amsterdam → Berlin →
Warsaw → Minsk → Moscow peaks at +7, +6, +10, +8, +7 and +13 h: about
50–70 km/h step by step, and nearer 40 km/h over the long hops, where
the peak is broad.

So a +6 h forecast can use the city 300–500 km upwind. A +48 h forecast
would need the weather some 2,000 km upwind — the Atlantic, for
anything west of Berlin, and outside the panel. The correlations are
small (r ≤ 0.24): a feature to add, not a forecast on its own.

**Question 7.** For the +6 h forecast in Berlin, which cities' rain
at the issue time would you give the model, and which for +48 h?

*Your answer:*

---

## 8. Other features vs future rain

The agent receives eight variables per city and hour, not just rain.
Take four at the issue time *t* — humidity and cloud cover, and how
each changed over the previous six hours — and look at the chance of
rain at *t* + 6 and *t* + 48 in each band:

In [ ]:
T = len(W)
t = slice(6, T - 48)                  # t - 6 and t + 48 both exist
h_, c_ = hum.to_numpy(), clouds.to_numpy()
rain_6h = rain.rolling(6).sum().to_numpy()
feats = pd.DataFrame({
    "humidity %": h_[t].ravel(),
    "clouds %": c_[t].ravel(),
    "humidity change 6 h": (h_[t] - h_[:T - 54]).ravel(),
    "clouds change 6 h": (c_[t] - c_[:T - 54]).ravel(),
    "rain last 6 h": rain_6h[t].ravel(),
    "wet now": W[t].ravel(),
})
wet_6 = W[12:T - 42].ravel()          # the hour t + 6
wet_48 = W[54:].ravel()               # the hour t + 48

BANDS = {
    "humidity %": [0, 60, 70, 80, 90, 100],
    "clouds %": [-1, 20, 40, 60, 80, 100],
    "humidity change 6 h": [-100, -15, -5, 5, 15, 100],
    "clouds change 6 h": [-101, -40, -10, 10, 40, 100],
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (name, edges) in zip(axes, BANDS.items()):
    band = pd.cut(feats[name], edges)
    p = pd.DataFrame({"+6 h": wet_6, "+48 h": wet_48}).groupby(
        band.to_numpy(), observed=True).mean()
    x = range(len(p))
    ax.plot(x, p["+6 h"], "o-", color="#2a78d6", lw=2, label="+6 h")
    ax.plot(x, p["+48 h"], "o-", color="#eb6834", lw=2,
            label="+48 h")
    ax.axhline(p_wet, color="#8c8b86", lw=1, ls=":")
    ax.set_xticks(x, [str(i) for i in p.index], rotation=35,
                  fontsize=7, ha="right")
    ax.set_title(name)
axes[0].set_ylabel("P(wet later)")
axes[0].legend()
plt.show()

One number per feature: the **AUC** of the feature as a score for
"wet later" — the probability that a random wet hour had a higher
value at *t* than a random dry one. 0.5 is useless, and an AUC below
0.5 is as informative as its mirror above.

In [ ]:
def auc(score, wet):
    ranks = pd.Series(score).rank().to_numpy()
    n1, n0 = wet.sum(), (~wet).sum()
    return (ranks[wet].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

print(pd.DataFrame({name: {"AUC +6 h": auc(feats[name], wet_6),
                           "AUC +48 h": auc(feats[name], wet_48)}
                    for name in feats}).T.round(3))

**Finding.** Cloud cover at the issue time says as much about rain at
+6 h as the rain of the last six hours (AUC 0.718 and 0.717): under 20%
cloud the chance of rain 6 hours later is 3%, over 80% it is 23%.
Humidity follows (0.659): 5% below 60%, 24% above 90%. Both are close
to useless at +48 h (0.576 and 0.527).

The changes are weak (0.513 and 0.553) and bent: rain at +6 h is most
likely when humidity did *not* change, and a rise in humidity makes it
*less* likely. Humidity rises every evening, when rain is rarest
(section 5): the change mostly measures the time of day.

**Question 8.** Humidity falls every morning as the air warms and
rises every evening. What does that do to "humidity change over 6
hours" as a feature, and what should sit beside it in the model?

*Your answer:*

---

## 9. Baselines, scored like the challenge

The challenge asks, for each city and each lead time *h* ∈ {6, 48},
for an **ensemble** of M plausible amounts x₁ … x_M (1 ≤ M ≤ 100), and
scores it against the observed rain y with the CRPS:

$$\mathrm{CRPS} = \frac{1}{M}\sum_i |x_i - y|
- \frac{1}{2M^2}\sum_i\sum_j |x_i - x_j|$$

In mm, lower is better, and it equals |x − y| when M = 1. The first
term rewards members near the outcome; the second rewards spread, so
a forecast cannot win by piling every member on one value.

The **reference** is the city's own last 48 hourly values used as 48
members. For each run the challenge computes, per lead time,

$$\mathrm{skill}_h = \frac{\overline{\mathrm{CRPS}}_{ref}
- \overline{\mathrm{CRPS}}_{agent}}{C_h[\mathrm{month}]}$$

floored at −1, with the means over the 45 cities and C_h a **fixed**
published table, `REFERENCE_CRPS`: the reference's mean CRPS in each
calendar month of the valid hour, over 2020–2024. The run's score is
the mean of the two skills; the leaderboard is the mean over runs.
The reference scores exactly 0. A perfect forecast scores the
reference's CRPS over C_h: about 1 on average, more in a wet run —
1 is not a ceiling.

In [ ]:
def crps(members, y):
    """CRPS of ensembles (..., M) against outcomes (...), in mm."""
    x = np.sort(np.maximum(members, 0.0), axis=-1)  # env clips at 0
    M = x.shape[-1]
    k = np.arange(1, M + 1)
    spread = (x * (2 * k - M - 1)).sum(axis=-1) / M ** 2
    return np.abs(x - y[..., None]).mean(axis=-1) - spread

print(crps(np.array([[0.3]]), np.array([1.0])))        # |0.3 - 1|
print(crps(np.array([[0.0, 0.0, 0.2, 1.0]]), np.array([0.2])))

The first line prints `[0.7]`. The second prints `[0.1]`: the members
are 0.3 mm from the outcome on average, and half their mean pairwise
distance is 0.2 mm. Check it by hand.

Now the test protocol. Forecasts are issued every 6 hours, at the UTC
hours the live challenge uses (its data arrive at 00, 06, 12 and 18
UTC, and the last complete hour is the issue time). Everything
fitted here is fitted on **2020–2024**;
the forecasts scored are those issued from **2025-01-01** to the last
one whose +48 h is in the file — 14 months no fit has seen.

In [ ]:
# C_h: the reference's mean CRPS (mm) per month of the valid hour,
# Jan..Dec, over 2020-2024 -- the challenge's published table.
REFERENCE_CRPS = {
    6: np.array([0.059, 0.063, 0.062, 0.069, 0.088, 0.088,
                 0.085, 0.084, 0.093, 0.098, 0.087, 0.077]),
    48: np.array([0.066, 0.069, 0.067, 0.075, 0.094, 0.094,
                  0.089, 0.090, 0.099, 0.105, 0.091, 0.082]),
}
HORIZONS = (6, 48)
ISSUE_HOURS = (5, 11, 17, 23)          # UTC, every 6 hours

split = rain.index.searchsorted(pd.Timestamp("2025-01-01", tz="UTC"))
issues = np.arange(split, T - max(HORIZONS))
issues = issues[np.isin(utc_hour[issues], ISSUE_HOURS)]
print(len(issues), "issue times,", rain.index[issues[0]], "->",
      rain.index[issues[-1]])

Four forecasts, each an array (issues, cities, members):

- **reference** — the last 48 hours, 48 members;
- **always 0** — one member, 0 mm;
- **persistence** — one member, the rain at the issue hour;
- **city × month climatology** — 100 members: the quantiles at levels
  0.005, 0.015, …, 0.995 of the city's rain in the calendar month of
  the hour being forecast, over 2020–2024.

In [ ]:
M = 100
levels = (np.arange(M) + 0.5) / M
train = r[:split]
clim_q = np.stack([np.quantile(train[month[:split] == m], levels,
                               axis=0).T for m in range(1, 13)])
print("climatology table:", clim_q.shape, "(month, city, member)")

reference = np.stack([r[i - 47:i + 1].T for i in issues])
results = {}
for h in HORIZONS:
    y = r[issues + h]                         # (issues, cities)
    m = month[issues + h] - 1                  # month of the valid hour
    forecasts = {
        "reference": reference,
        "always 0": np.zeros(y.shape + (1,)),
        "persistence": r[issues][..., None],
        "city x month climatology": clim_q[m],
    }
    ref_run = crps(reference, y).mean(axis=1)  # one value per run
    for name, f in forecasts.items():
        run = crps(f, y).mean(axis=1)
        c_h = REFERENCE_CRPS[h][m]
        skill = np.maximum(-1, (ref_run - run) / c_h)
        results.setdefault(name, {})[f"CRPS {h}h"] = run.mean()
        results[name][f"skill {h}h"] = skill.mean()

table = pd.DataFrame(results).T
table["score"] = table[["skill 6h", "skill 48h"]].mean(axis=1)
print(table.round(4))

**Finding.** Over 1,700 forecasts issued from 2025-01-01 to
2026-03-01:

- **Persistence** is a disaster (score −0.50). One member at the
  current value is wrong in full whenever rain starts or stops, and a
  single member has no spread to soften it: its CRPS is its absolute
  error.
- **Always 0** scores 0.001 — about the reference, as the scoring
  intends: it beats the reference at +48 h (+0.035) and loses at +6 h
  (−0.032), where the recent hours do carry information.
- **The city × month climatology** — 100 quantiles from 2020–2024, and
  no input at all — scores **+0.057**: +0.026 at +6 h and +0.089 at
  +48 h. It wins most where the reference is weakest: 48 hours of one
  city is a thin, stale sample of its climate.

**Question 9.** "Always 0" has a lower CRPS than the reference at
+48 h, and yet no one should submit it. What does it get wrong, and
which term of the CRPS says so?

*Your answer:*

---

## 10. What a good agent should use

- **A climatology first.** Per city and per month, for both horizons.
  It scores +0.057 with no input, and it *is* most of the +48 h
  forecast (section 6: ×0.92 after a dry hour).
- **The right shape.** A spike at 0 and a long tail (section 3): a
  probability of rain, times a distribution of wet-hour amounts that
  is heavier in summer and autumn (section 4). A hurdle model, whose
  quantiles are the members.
- **The state at the issue time, for +6 h.** Rain in the last hours,
  cloud cover and humidity (AUC 0.66–0.72, section 8), and rain in the
  city 300–500 km upwind (section 7).
- **The solar hour, in summer.** Convection peaks in the afternoon,
  steeply in the east and south (section 5). But splitting every city
  × month cell by hour leaves 1/24 of the data in each: measure before
  keeping it. The agent notebook does, and finds no gain on its own.
- **Not:** a single member (persistence, −0.50), or a 6-hour change
  without the time of day beside it (section 8).

**Question 10.** Write the three features you would add first to the
climatology, one for each horizon and one for both, and the section
of this notebook that justifies each.

*Your answer:*

The next notebook turns the climatology row of section 9 into a
working agent and submits it:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-agent.ipynb)